# LSTNet: long- and short-term patterns in multivariate time series

Electricity use, road traffic and solar power repeat every day and every week, and hundreds of series move together. Can a neural network that looks both at the last few hours and at the same hour on earlier days forecast them better than an autoregression? Lai et al. (2018) built LSTNet for it and compared it with eight methods on four datasets (Table 2, p. 101). Here their LSTNet-Skip, translated from their PyTorch code, is trained on the exchange rates at horizon 3, the cell of Table 2 quickest to train, and printed beside the article's values.

The data are the article's four datasets, as the authors prepared them (laiguokun/multivariate-time-series-data): read into memory and not saved, since the repository has no licence. LSTNet-Skip comes from the LSTNet package, of Compose › New Methods: no Julia package has the authors' model.

In [ ]:
Pollis.packages("LSTNet" => "https://github.com/Trumpingtons/LSTNet.jl", "CSV", "Tables", "Plots")

## Set-up

`fetchdata` reads a dataset into memory, one column per series; `using LSTNet` for `Split` (the article's 60/20/20 split, each series divided by its largest value), `forecast`, `evaluate` (RSE and CORR), `SETTINGS` and `table2`; and `replicate`, which prints each value in full, rounded to the published precision, beside the published value, and how many decimals they share.

In [ ]:
using LSTNet, CSV, Tables, Downloads, Plots, Printf, Statistics

"""
	replicate(title, rows)

Print each Julia value in full, rounded to the published precision and beside the published value, with the number of decimals they share.
Each row is `(label, value, published)`, with `published` written as in the source: its decimals give the published precision.
"""
function replicate(title, rows)
	@printf("\n%s\n%-26s%16s%14s%14s\n", title, "", "Julia", "rounded", "published")
	for (label, value, published) in rows
		p = parse(Float64, replace(published, "," => ""))
		d = occursin('.', published) ? length(split(published, '.')[end]) : 0  # decimals as published
		k = findlast(j -> round(value, digits=j) == round(p, digits=j), 0:d)  # position in 0:d
		@printf("%-26s%16.8f%14s%14s  %s\n", label, value, Printf.format(Printf.Format("%.$(d)f"), value), published, k === nothing ? "DIFFERENT" : "same to $(k - 1) decimals")
	end
end

const DATA = "https://raw.githubusercontent.com/laiguokun/multivariate-time-series-data/7f402f185cc2435b5e66aed13a3b560ed142e023/"
"The dataset at `path` in the authors' data repository, read into memory (never saved): one column per series."
fetchdata(path) = Tables.columntable(CSV.File(take!(Downloads.download(DATA * path, IOBuffer())); header=false))

exchange = fetchdata("exchange_rate/exchange_rate.txt.gz");  # daily, 8 currencies, 1990-2016

## 1. The data

The four datasets of the article (Section 4.3): Electricity, the hourly consumption of 321 clients; Traffic, the hourly occupancy of 862 freeway sensors; Solar-Energy, the output of 137 plants every 10 minutes; and Exchange-Rate, the daily rates of 8 currencies. The three larger ones are 56 MB gzipped together and take a while to read. The autocorrelation, averaged over each dataset's series, shows the patterns the network is built for: peaks a day apart, and a week apart, in Electricity and Traffic, a daily cycle in Solar-Energy (144 steps of 10 minutes), and none in the exchange rates, which only drift.

In [ ]:
electricity = fetchdata("electricity/electricity.txt.gz");
traffic = fetchdata("traffic/traffic.txt.gz");
solar = fetchdata("solar-energy/solar_AL.txt.gz");

"The autocorrelation at each of `lags`, averaged over the series of `x` that are not constant."
acf(x, lags) = [mean(cor(view(x, 1:size(x, 1)-k, j), view(x, 1+k:size(x, 1), j)) for j in axes(x, 2) if std(view(x, :, j)) > 0) for k in lags]

datasets = [("Electricity, hourly", electricity), ("Traffic, hourly", traffic), ("Solar-Energy, every 10 minutes", solar), ("Exchange-Rate, daily", exchange)]
for (name, t) in datasets
	@printf("%-32s %6d times, %4d series\n", name, size(Tables.matrix(t))...)
end
lags = 1:24*8
plot([plot(lags, acf(Tables.matrix(t), lags), label="", title=name, titlefontsize=9, ylims=(-0.2, 1)) for (name, t) in datasets]..., layout=(2, 2), xlabel="lag", size=(800, 500))

## 2. LSTNet-Skip

For a window of the last 168 observations of every series, the network adds two forecasts: a neural part (a convolution of 6 steps across all the series, a GRU over its outputs, and a second GRU over the steps 24 apart, the *recurrent skip*, for the same hour on earlier days) and a highway, an autoregression on the last 24 values of each series, which keeps the scale the neural part is insensitive to. The training is the authors' code's: absolute error, Adam, batches of 128, the gradient's norm clipped at 10, 100 epochs, and the epoch with the smallest validation RSE kept. `SETTINGS` holds each dataset's settings from the authors' scripts; for the exchange rates (`stock.sh`), hidden sizes of 50 and no output function.

In [ ]:
SETTINGS["exchange"]

## 3. Exchange-Rate, horizon 3

**This cell trains a network for 100 epochs: about 30 minutes on a Mac's GPU, and longer on most CPUs.** It trains on a GPU if its package is loaded first (uncomment the line for yours), and on the CPU otherwise. The training is saved after every epoch to `LSTNet-exchange-3.checkpoint`, in the notebook's folder: stop the cell at any time, and run it again to continue from the last epoch; once finished, running it again takes seconds.

Table 2 reports one training per cell, with a seed and tuning that are not known, and the random numbers of Julia and PyTorch differ, so the values cannot match to every digit: they should be close.

In [ ]:
# using Metal   # or CUDA, AMDGPU, oneAPI: the training then runs on that GPU
s = Split(exchange, 3)  # forecasts 3 days ahead
f = forecast(LSTNetSkip(SETTINGS["exchange"]), s; checkpoint="LSTNet-exchange-3.checkpoint")
e = evaluate(s, f.test)
@printf("Epoch kept: %d of 100; training: %.0f s\n", f.epoch, f.seconds)
replicate("Table 2, Exchange-Rate, horizon 3, LSTNet-Skip (p. 101)", [("RSE", e.rse, "0.0226"), ("CORR", e.corr, "0.9735")])

The forecasts of the test period, the last 20 per cent of the days, beside the rates, for the first currency:

In [ ]:
days = s.test
plot(days, s.raw[days, 1], label="exchange rate", lw=1.5)
plot!(days, f.test[:, 1] .* s.scale[1], label="LSTNet-Skip, 3 days ahead", lw=1, xlabel="day", ylabel="rate", size=(800, 350))

## 4. Against the simpler methods

Table 2's Exchange-Rate columns for LSTNet-Skip, the autoregression (AR) and linear ridge regression (LRidge), with the simplest forecast of all beside them: tomorrow's rate is today's (the last value observed, `horizon` days before). The baselines are the article's; they are not in the authors' code, so they are not run here.

In [ ]:
horizons = (3, 6, 12, 24)
article = [("LSTNet-Skip", [0.0226, 0.0280, 0.0356, 0.0449], [0.9735, 0.9658, 0.9511, 0.9354]),
	("AR", [0.0228, 0.0279, 0.0353, 0.0445], [0.9734, 0.9656, 0.9526, 0.9357]),
	("LRidge", [0.0184, 0.0274, 0.0419, 0.0675], [0.9788, 0.9722, 0.9543, 0.9305])]
@printf("%-30s%s\n", "RSE / CORR, horizon", join([@sprintf("%16d", h) for h in horizons]))
for (name, r, c) in article
	@printf("%-30s%s\n", name * " (article)", join([@sprintf("%8.4f%8.4f", r[j], c[j]) for j in eachindex(horizons)]))
end
naive = [let s = Split(exchange, h); evaluate(s, s.data[s.test .- h, :]) end for h in horizons]
@printf("%-30s%s\n", "last value (Julia)", join([@sprintf("%8.4f%8.4f", l.rse, l.corr) for l in naive]))

On the exchange rates the network has no daily or weekly pattern to find, and the article's linear methods do as well or better: at horizon 3 the autoregression's RSE is 0.0228 and the ridge regression's 0.0184, against LSTNet-Skip's 0.0226. The network's advantage is on the three datasets with the patterns of section 1.

## 5. The rest of Table 2

The other 15 cells, the four horizons of each dataset, take far longer: Solar-Energy, Electricity and Traffic have 137, 321 and 862 series, and each cell is 100 epochs. They are meant for a machine with an NVIDIA GPU. `table2` saves each finished horizon in `folder` (`DATASET-HORIZON.toml`) and the training after every epoch, so it can be stopped and continued; set `RUN_ALL = true` to run them.

In [ ]:
RUN_ALL = false
if RUN_ALL
	for (name, t) in [("exchange", exchange), ("solar", solar), ("electricity", electricity), ("traffic", traffic)]
		r = table2(t, name; folder="LSTNet-table2")  # RSE (row 1) and CORR (row 2) at horizons 3, 6, 12, 24
		@printf("%-12s RSE  %s\n%-12s CORR %s\n", name, join([@sprintf("%8.4f", x) for x in r[1, :]]), "", join([@sprintf("%8.4f", x) for x in r[2, :]]))
	end
end

## 6. What differs, and why

- **The values are close, not equal.** A single training from scratch with the authors' settings: on this Mac's GPU it gave RSE 0.0256 and CORR 0.9706, against the article's 0.0226 and 0.9735 (epoch 97 kept, 1729 s). The article reports one run per cell; its seed is not known (54321 is the code's default, which the package uses, and the scripts set none) and its hidden sizes were tuned by a grid search the scripts do not record. Julia and PyTorch draw different random numbers from the same seed, and a GPU and a CPU round differently, so each machine gives a somewhat different value.
- **That the model is the authors' is shown by the cross-tests, not by this cell.** Given the authors' initial weights, the package's forecasts agree with their code's to 1.8e-7; given the same weights and batches, three epochs of training agree to 1.1e-6. Trained for 100 epochs, even from identical starts the two codes part, from the rounding of a single number: the vignette (Compose › New Methods) shows how, and that it moves the authors' own result as much.
- **The code, not the paper.** Where they differ, the package follows the authors' code: GRUs with tanh, not the ReLU of Section 3.3; ⌊(168 - 6) / 24⌋ = 6 skip periods (the code's division without rounding fails with its own defaults); the absolute-error loss always (`--L1Loss False` is read as true); and the RSE with the code's standard deviation.
- **FluxArchitectures.jl's `LSTnet` is a different model.** It forecasts one series, with a convolution as long as the window and a GRU carried across samples, so it cannot give Table 2.

**Now in Compose › New Methods.** No Julia package has LSTNet as its authors wrote it. The New Methods package **LSTNet**, used here, has LSTNet-Skip, the article's split and its RSE and CORR as documented functions, on any GPU or the CPU, with training that can be stopped and continued, tested against the authors' code. Find it, and its vignette, in **Compose › New Methods**.

## References

- Lai, G., Chang, W.-C., Yang, Y. and Liu, H. (2018). Modeling long- and short-term temporal patterns with deep neural networks. In *SIGIR '18: The 41st International ACM SIGIR Conference on Research and Development in Information Retrieval*, 95-104. doi:10.1145/3209978.3210006
- Code: github.com/laiguokun/LSTNet, commit 093c4dc.
- Data: github.com/laiguokun/multivariate-time-series-data, commit 7f402f1, read into memory and not saved, since the repository has no licence.